# LoRA Applied: How It Fits

In [1]:
!pip install -q transformers peft datasets accelerate

In [2]:
from datasets import load_dataset

# Download the IMDb dataset
dataset = load_dataset("imdb")

# Verify we have train and test splits
print(f"Training samples: {len(dataset['train'])}")
print(f"Test samples: {len(dataset['test'])}")

Training samples: 25000
Test samples: 25000


In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    result = tokenizer(batch["text"], padding="max_length", truncation=True, max_length=512)
    result["labels"] = batch["label"]  # Keep the label for training
    return result

In [4]:
dataset = dataset.map(tokenize, batched=True)

# Set format for PyTorch

dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [5]:
from transformers import AutoModelForSequenceClassification


model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)

# Count trainable parameters in baseline

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable} / {total} ({100*trainable/total:.2f}%)")
# Expected: Trainable: ~67M / 67M (100%)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable: 66955010 / 66955010 (100.00%)


# Efficient Fine-Tuning with LoRA: Training and Evaluation in Practice

In [6]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,                              # Rank: smaller = fewer params
    lora_alpha=16,                    # Scaling factor for LoRA updates
    target_modules=["q_lin", "v_lin"], # DistilBERT attention layers
    lora_dropout=0.1,                 # Regularization
    bias="none",                      # Don't add bias parameters
    task_type="SEQ_CLS"               # Sequence classification task
)


In [7]:
# Attach LoRA adapters to the model

lora_model = get_peft_model(model, lora_config)

# Show the dramatic reduction in trainable parameters

lora_model.print_trainable_parameters()


trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",      # Check performance after each epoch
    num_train_epochs=1,               # Just 1 epoch for demo
    per_device_train_batch_size=8,   # 8 examples per batch
    per_device_eval_batch_size=8,
    logging_steps=50,                 # Log every 50 steps
    save_steps=2000,                  # Save checkpoint every 2000 steps
    learning_rate=5e-4,               # Higher than typical (faster demo)
    weight_decay=0.01,                # L2 regularization
)

# Create small train/eval subsets

train_subset = dataset["train"].shuffle(seed=42).select(range(2000))
eval_subset = dataset["test"].shuffle(seed=42).select(range(500))

# Initialize the Trainer

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=train_subset,
    eval_dataset=eval_subset,
)

# Run the training

print("Starting training with LoRA adapters...")
trainer.train()


Starting training with LoRA adapters...


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


In [ ]:
text = "This movie was an absolute masterpiece. Brilliant performances!"

# Tokenize the input text

inputs = tokenizer(text, return_tensors="pt")

# Move inputs to same device as models

device = next(model.parameters()).device
inputs = {k: v.to(device) for k, v in inputs.items()}

# Get predictions from both models

with torch.no_grad():  # Don't compute gradients during inference
    baseline_logits = model(**inputs).logits
    lora_logits = lora_model(**inputs).logits

# Convert logits to probabilities

baseline_probs = baseline_logits.softmax(-1)[0].cpu().numpy()
lora_probs = lora_logits.softmax(-1)[0].cpu().numpy()

print(f"Input: {text}")
print(f"\nBaseline predictions:")
print(f"  Negative: {baseline_probs[0]:.1%}")
print(f"  Positive: {baseline_probs[1]:.1%}")
print(f"\nLoRA fine-tuned predictions:")
print(f"  Negative: {lora_probs[0]:.1%}")
print(f"  Positive: {lora_probs[1]:.1%}")